# AdWords Problem

## Learning Objectives

1. **Formulate** the AdWords problem with budgets and bids
2. **Derive** the BALANCE algorithm and its penalty function ψ(f) = 1 − e^{f−1}
3. **Prove** the competitive ratio of 1 − 1/e for equal bids
4. **Compare** BALANCE against greedy (highest-bid) in simulation

## Problem Setup

**Advertisers** $A = \{a_1,\ldots,a_m\}$ each with:
- Budget $b_i$: total amount willing to spend
- Bid $x_{iq}$: amount paid if matched to query $q$

**Queries** $q_1, q_2, \ldots$ arrive online.

**Assignment rule:** assign query $q$ to one eligible advertiser; that advertiser pays $x_{iq}$ and their budget decreases.

**Objective:** maximize total revenue $\sum_{(i,q)\text{ matched}} x_{iq}$

**Constraint:** total payments to advertiser $i$ cannot exceed $b_i$

This is the core problem behind Google's search ad auction.

## Greedy Baseline

**Greedy (highest-bid):** assign query $q$ to the advertiser with the highest bid $x_{iq}$ who still has budget remaining.

**Problem:** greedy ignores budget depletion rate. A high-bidding advertiser with a small remaining budget gets priority, depleting early and missing later high-value queries.

**Competitive ratio of greedy:** can be arbitrarily bad in the worst case.

Example: two advertisers, budget = 1 each. Advertiser A bids 1 on every query, Advertiser B bids 0.5. Greedy always picks A; A's budget depletes after 1 query; B never gets a chance even on queries that only B can match.

## BALANCE Algorithm

**Key insight:** penalize advertisers who have already spent a large fraction of their budget.

Define the fraction spent: $f_i = \frac{\text{spent}_i}{b_i} \in [0, 1]$

**Penalty function:** $\psi(f) = 1 - e^{f-1}$

- $\psi(0) = 1 - 1/e \approx 0.632$ (fresh budget — high priority)
- $\psi(1) = 0$ (budget exhausted — never assigned)

**BALANCE rule:** assign query $q$ to advertiser $i^*$ maximizing:

$$i^* = \arg\max_{i \text{ eligible, has budget}} x_{iq} \cdot \psi(f_i)$$

This interpolates between: "bid-optimal" (when all budgets full) and "avoid depleted budgets" (as budgets shrink).

In [1]:
import math

def balance_algorithm(advertisers, budgets, bids, query_sequence):
    """
    BALANCE algorithm for AdWords.
    advertisers: list of advertiser IDs
    budgets: dict a -> budget
    bids: dict (a, query) -> bid (0 if ineligible)
    query_sequence: list of query IDs
    Returns: total revenue, assignment dict
    """
    remaining = dict(budgets)
    spent = {a: 0.0 for a in advertisers}
    assignment = {}
    revenue = 0.0

    def psi(f):
        return 1 - math.exp(f - 1)

    for q in query_sequence:
        best_score = -1
        best_adv = None
        for a in advertisers:
            bid = bids.get((a, q), 0)
            if bid > 0 and remaining[a] >= bid:
                f = spent[a] / budgets[a]
                score = bid * psi(f)
                if score > best_score:
                    best_score = score
                    best_adv = a
        if best_adv is not None:
            payment = bids[(best_adv, q)]
            assignment[q] = (best_adv, payment)
            remaining[best_adv] -= payment
            spent[best_adv] += payment
            revenue += payment

    return revenue, assignment

def greedy_algorithm(advertisers, budgets, bids, query_sequence):
    remaining = dict(budgets)
    assignment = {}
    revenue = 0.0
    for q in query_sequence:
        best_bid = -1; best_adv = None
        for a in advertisers:
            bid = bids.get((a,q),0)
            if bid > 0 and remaining[a] >= bid and bid > best_bid:
                best_bid = bid; best_adv = a
        if best_adv is not None:
            assignment[q] = (best_adv, best_bid)
            remaining[best_adv] -= best_bid
            revenue += best_bid
    return revenue, assignment

# Simulation
import random
random.seed(42)
advertisers = ['A','B','C']
budgets = {'A':10,'B':10,'C':10}
queries = [f'q{i}' for i in range(30)]
bids = {}
for a in advertisers:
    for q in queries:
        bids[(a,q)] = round(random.uniform(0.5,2.0),2) if random.random()<0.6 else 0

rev_bal, _ = balance_algorithm(advertisers, budgets, bids, queries)
rev_gr, _  = greedy_algorithm(advertisers, budgets, bids, queries)
print(f"BALANCE revenue: {rev_bal:.2f}")
print(f"Greedy revenue:  {rev_gr:.2f}")
print(f"BALANCE / Greedy: {rev_bal/max(rev_gr,0.01):.3f}")

BALANCE revenue: 29.30
Greedy revenue:  28.23
BALANCE / Greedy: 1.038


## Competitive Ratio = 1 − 1/e

**Theorem (Mehta et al. 2007):** BALANCE achieves competitive ratio $1 - 1/e$ for AdWords when:
- All bids are small relative to budgets ($x_{iq} \ll b_i$)
- Budgets are equal across advertisers

**Proof sketch (equal bids, unit budgets):**

Consider an advertiser $i$ with fraction $f$ of budget spent. When a query arrives that *optimal offline* assigns to $i$ but BALANCE assigns elsewhere:

The offline gain is 1 unit. BALANCE chose another advertiser $j$ with $\psi(f_j) \geq \psi(f_i)$ (higher score) meaning $j$ had less spent budget — BALANCE was rational.

Summing over all lost opportunities and integrating $\psi(f)$ from 0 to 1:

$$\int_0^1 \psi(f)\,df = \int_0^1 (1-e^{f-1})\,df = 1 - (e^0 - e^{-1}) = 1 - \frac{e-1}{e} = \frac{1}{e}$$

So the total loss from BALANCE vs optimal is at most $1/e$ of OPT, giving CR $= 1 - 1/e$.

In [2]:
# Monte Carlo CR estimation
import random, math

def offline_optimal_adwords(advertisers, budgets, bids, queries):
    """Greedy approximation to offline optimal (for comparison)."""
    # Sort all (a,q) pairs by bid descending; assign greedily
    all_pairs = [(bids[(a,q)], a, q) for a in advertisers for q in queries if bids.get((a,q),0)>0]
    all_pairs.sort(reverse=True)
    remaining = dict(budgets); assigned_q = set(); revenue = 0.0
    for bid, a, q in all_pairs:
        if q not in assigned_q and remaining[a] >= bid:
            assigned_q.add(q); remaining[a] -= bid; revenue += bid
    return revenue

trials = 200; cr_ratios = []
for seed in range(trials):
    rng = random.Random(seed)
    advs = ['A','B','C','D']
    bdg = {a: 5.0 for a in advs}
    qs = [f'q{i}' for i in range(20)]
    bs = {(a,q): round(rng.uniform(0.3,1.0),2) if rng.random()<0.5 else 0 for a in advs for q in qs}
    rev_bal,_ = balance_algorithm(advs,bdg,bs,qs)
    rev_opt   = offline_optimal_adwords(advs,bdg,bs,qs)
    if rev_opt>0: cr_ratios.append(rev_bal/rev_opt)

print(f"Estimated avg CR over {trials} trials: {sum(cr_ratios)/len(cr_ratios):.3f}")
print(f"Theoretical guarantee:                  {1-1/math.e:.3f}")
print(f"Min CR observed: {min(cr_ratios):.3f}")

Estimated avg CR over 200 trials: 0.998
Theoretical guarantee:                  0.632
Min CR observed: 0.932


## Summary

| Algorithm | CR | Notes |
|-----------|-----|-------|
| Greedy (highest bid) | Unbounded loss | Ignores budget state |
| BALANCE | 1 − 1/e ≈ 0.632 | Penalty on spent fraction |
| Offline optimal | 1.0 | Sees all queries |

The penalty function $\psi(f) = 1 - e^{f-1}$ is the unique function that achieves the optimal CR of $1-1/e$.
It balances revenue (high bid) against fairness (don't deplete budgets too fast).